# Read your Kotar model in Python

This notebook connects to the SysML v2 service in Kotar desktop, lists your projects, gets the elements of your latest commit, and prints the model. It uses [`sysml-api-client`](https://pypi.org/project/sysml-api-client/) and the [`sysml`](https://pypi.org/project/sysml/) SDK.

**Setup** (Python 3.10 or newer), from the `python-notebook` folder:

```bash
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
jupyter lab notebooks/
```

**Before you start**, send a model to the service: in Kotar desktop, run **SysML: Commit Model to SysMLv2 Service…** from the Command Palette, choose **Mesh SysMLv2**, and pick or create a project. A Git commit alone doesn't send the model.

## 1. Connect

While Kotar desktop runs, it writes the service's port and an access token to `mesh-owner.json`. The token changes every time Kotar starts, so re-run this cell after restarting Kotar.

The path below is for macOS. On Windows the file should be `%APPDATA%\Kotar\mesh-owner.json`; on Linux, `~/.config/Kotar/mesh-owner.json`.

In [ ]:
import json
from pathlib import Path

from sysmlv2_client import SysMLV2Client

kotar = json.loads((Path.home() / "Library/Application Support/Kotar/mesh-owner.json").read_text())
url = f"http://127.0.0.1:{kotar['ports']['sysmlv2']}"

client = SysMLV2Client(base_url=url, bearer_token="Bearer unused")
client._session.headers.pop("Authorization")
client._session.headers["x-sysmlv2-svc-token"] = kotar["sessionToken"]
print("Connected to", url)

## 2. List projects

Kotar names each project after its workspace.

In [ ]:
projects = client.get_projects()
for p in projects:
    print(p["name"], "-", p["@id"])

assert projects, "No projects yet. Commit a model in Kotar first."
project = projects[0]  # the newest
project_id = project["@id"]
print("\nUsing", project["name"])

## 3. Find the latest commit

Each Git branch in Kotar has a branch with the same name here. Its head is your latest commit.

In [ ]:
branches = client.list_branches(project_id)
print(branches)
commit_id = branches[0]["head"]["@id"]  # a branch's head is its latest commit
print(commit_id)

## 4. Get the elements

`client.list_elements` returns the elements at a commit.

In [ ]:
elements = client.list_elements(project_id, commit_id)
print(len(elements), "elements")
print(elements)

Elements are paged. To get many:

In [ ]:
def get_elements(project_id, commit_id, page_size=1000):
    """Every element at a commit. The service sends them in pages; each page links to the next."""
    elements = []
    next_page = f"/projects/{project_id}/commits/{commit_id}/elements?page[size]={page_size}"
    while next_page:
        response = client._session.get(url + next_page)
        response.raise_for_status()
        elements += response.json()
        next_page = response.links.get("next", {}).get("url")
    return elements

get_elements(project_id, commit_id)

## 5. Print the model with the SysML SDK

The SDK turns each element into a class named after its metaclass, with the specification's property names (`ownedMember`, `declaredName`, ...). The first run downloads the standard library (about 11 MB).

In [ ]:
from sysml import Model, PayloadLibrary, standard_library_json

library = PayloadLibrary(json.loads(Path(standard_library_json()).read_text()))
model = Model.from_full_json(elements, library=library)

def print_tree(element, depth=0):
    for relationship in element.ownedRelationship:
        for child in relationship.ownedRelatedElement:
            print("    " * depth + f"{type(child).__name__} {child.declaredName or ''}")
            print_tree(child, depth + 1)

for root in model.roots:
    print_tree(root)